In [1]:
import sys
sys.path.insert(0, r"C:\Users\USER\Downloads\judging_gemma")
from judging_gemma.config import *
from judging_gemma.judging_utils import load_and_fix_gt, load_clinical_gt, parse_json_response, attach_cleaned_text

In [2]:
import json

import re

import time

import os

import pandas as pd

from openai import OpenAI

from dotenv import load_dotenv



load_dotenv()

API_KEY = os.getenv("GLM_API_KEY")



INPUT_FILE   = r"C:\Users\USER\Downloads\alterllama_dsk_8b_results_with_pd.csv"

MIMIC_FIXED  = r"C:\Users\USER\Downloads\neweraugmented_mimic_admission_only.csv"

OUTPUT_0SHOT = r"C:\Users\USER\Downloads\llama8b_glmjudge_0shot.csv"

OUTPUT_1SHOT = r"C:\Users\USER\Downloads\llama8b_glmjudge_1shot.csv"

MODEL = GLM_MODEL



client = OpenAI(

    api_key=API_KEY,

    base_url = GLM_BASE_URL

)



SYSTEM_PROMPT = """You are a clinical diagnostic auditor. Score TWO accuracy dimensions.



D1_ICD: How well does the model diagnosis match the ICD-coded ground truth (LONG_TITLE)?

D1_CLINICAL: How well does it match the clinician-level principal diagnosis (CLINICAL_GT)?



Both scored 0-3:

3=exact/equivalent, 2=correct category wrong specificity, 1=mentioned as differential only, 0=wrong



Return ONLY JSON:

{

  "d1_icd_accuracy": <0-3>,

  "d1_clinical_accuracy": <0-3>,

  "model_dx": "<extracted diagnosis>",

  "reasoning": "<one sentence>"

}"""



def judge_accuracy(row):

    prompt = (

        f"Model Diagnosis: {row['predicted_pd']}\n"

        f"ICD Ground Truth (LONG_TITLE): {row['LONG_TITLE']}\n"

        f"Clinical Ground Truth: {row['clinical_gt']}"

    )



    try:

        response = client.chat.completions.create(

            model=MODEL,

            messages=[

                {"role": "system", "content": SYSTEM_PROMPT},

                {"role": "user",   "content": prompt}

            ],

            max_tokens=3072,

            temperature=0.0

        )

        content = response.choices[0].message.content

        if not content or not content.strip():

            raise ValueError("Empty response from API")

        clean_content = re.sub(r"```json|```", "", content).strip()

        return json.loads(clean_content)

    except Exception as e:

        return {

            "d1_accuracy": -1,

            "model_dx": "",

            "reasoning": str(e)

        }

In [6]:
CLINICAL_GT_FILE = r"C:\Users\USER\Downloads\judging_gemma\judging_gemma\clinical_ground_truth.csv"

df = pd.read_csv(INPUT_FILE)

# Fix ground truth columns
df = load_and_fix_gt(df)

# Merge clinical_gt directly (hardcoded path avoids __file__ resolution issues in notebooks)
cgt = pd.read_csv(CLINICAL_GT_FILE)
cgt.columns = cgt.columns.str.strip()
cgt = cgt[['HADM_ID', 'clinical_gt']].drop_duplicates('HADM_ID')
df = df.drop(columns=[c for c in ['clinical_gt'] if c in df.columns])
df = df.merge(cgt, on='HADM_ID', how='left')

print(f"Loaded {len(df)} rows | clinical_gt merged: {df['clinical_gt'].notna().sum()} non-null")

# Keep one row per (HADM_ID, condition) — removes duplicate model runs
df_baseline = (
    df[df['condition'].isin(['0-shot', '1-shot'])]
    .drop_duplicates(subset=['HADM_ID', 'condition'], keep='first')
    .copy()
)

n_0shot = (df_baseline['condition'] == '0-shot').sum()
n_1shot = (df_baseline['condition'] == '1-shot').sum()
print(f"Processing {len(df_baseline)} baseline rows: {n_0shot} x 0-shot, {n_1shot} x 1-shot")

results = []
for i, (_, row) in enumerate(df_baseline.iterrows()):
    condition = row['condition']
    print(f"[{i+1}/{len(df_baseline)}] [{condition}] HADM {row['HADM_ID']}...")

    evaluation = judge_accuracy(row)
    print(evaluation)

    combined = {**row.to_dict(), **evaluation}
    results.append(combined)

    time.sleep(0.5)

results_df = pd.DataFrame(results)

df_0shot = results_df[results_df['condition'] == '0-shot']
df_1shot = results_df[results_df['condition'] == '1-shot']

df_0shot.to_csv(OUTPUT_0SHOT, index=False)
df_1shot.to_csv(OUTPUT_1SHOT, index=False)

print(f"\n--- Results ---")
print(f"0-shot  ({len(df_0shot)} rows) → {OUTPUT_0SHOT}  |  avg d1_icd: {df_0shot['d1_icd_accuracy'].mean():.2f}  avg d1_clinical: {df_0shot['d1_clinical_accuracy'].mean():.2f}")
print(f"1-shot  ({len(df_1shot)} rows) → {OUTPUT_1SHOT}  |  avg d1_icd: {df_1shot['d1_icd_accuracy'].mean():.2f}  avg d1_clinical: {df_1shot['d1_clinical_accuracy'].mean():.2f}")





Loaded 35 rows | clinical_gt merged: 35 non-null
Processing 10 baseline rows: 5 x 0-shot, 5 x 1-shot
[1/10] [0-shot] HADM 197345...
{'d1_icd_accuracy': 2, 'd1_clinical_accuracy': 3, 'model_dx': 'Metastatic Breast Cancer', 'reasoning': 'The model exactly matches the clinical ground truth, but for the ICD ground truth it identifies the primary malignancy rather than the specific secondary metastatic site, representing the correct general category but wrong specificity.'}
[2/10] [1-shot] HADM 197345...
{'d1_icd_accuracy': 2, 'd1_clinical_accuracy': 3, 'model_dx': 'Metastatic Breast Carcinoma, Unspecified (Not Otherwise Specified)', 'reasoning': 'The model diagnosis exactly matches the clinical ground truth of metastatic breast cancer, but for the ICD ground truth it identifies the primary malignancy with metastasis rather than specifying the secondary site (retroperitoneum and peritoneum).'}
[3/10] [0-shot] HADM 176830...
{'d1_icd_accuracy': 3, 'd1_clinical_accuracy': 3, 'model_dx': 'Gast